In [1]:
from eth_account import Account
from web3 import Web3
from scripts import config as c
from scripts.web3_client_handler import get_w3_client

In [ ]:
w3 = get_w3_client()

## 1. Basic commands

In [ ]:
w3.is_connected()

In [ ]:
w3.eth.get_block("latest")

In [ ]:
c.PK_DEPLOYER.key, c.PK_DEPLOYER.address

## 2. Contract interactions

In [ ]:
address = c.registry_contract_address
with open("contracts/GlobalRegistry.sol/GlobalRegistry.abi", "r") as f:
    abi = "".join(f.read().splitlines())
global_registry = w3.eth.contract(address=address, abi=abi)

In [ ]:
global_registry.functions.owner().call()

In [ ]:
global_registry.functions.exists("Wholesale NOK").call()

In [ ]:
## helper functions


def _get_contract_symbol(contract):
    return contract.functions.symbol().call()


def _do_contract_tx(contract, function_name, tx_args, caller_key):
    print("Sending new tx: " + function_name)
    caller = Account().from_key(caller_key).address
    transaction_data = contract.functions[function_name](*tx_args).build_transaction(
        {
            "chainId": w3.eth.chain_id,
            "chainId": 2018,
            "gas": 2000000,
            "gasPrice": w3.eth.gas_price,
            "nonce": w3.eth.get_transaction_count(caller),
        }
    )

    signed_txn = w3.eth.account.sign_transaction(
        transaction_data, private_key=caller_key
    )
    tx_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction)
    tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    print(tx_receipt)


_cct_from_caller_role = Web3.keccak(text="CCT_FROM_CALLER_ROLE")
_transfer_from_role = Web3.keccak(text="TRANSFER_FROM_ROLE")

In [ ]:
address = global_registry.functions.getContract("Wholesale NOK").call()
with open("contracts/Wnok.sol/Wnok.abi", "r") as f:
    abi = "".join(f.read().splitlines())
wnok = w3.eth.contract(address=address, abi=abi)

In [ ]:
wnok.functions.decimals().call()

### 2.1 TBD Interactions

In [ ]:
## helper functions


def _get_tbd_contract(contractName: str):
    address = global_registry.functions.getContract(contractName).call()
    with open("contracts/Tbd.sol/Tbd.abi", "r") as f:
        abi = "".join(f.read().splitlines())
    return w3.eth.contract(address=address, abi=abi)

In [ ]:
## get TBD Nordea contract

tbd_nordea = _get_tbd_contract("TBD Nordea")

# print TBD balance of Alice (retail investor at Nordea bank)
(
    str(tbd_nordea.functions.balanceOf(c.PK_ALICE_TBD.address).call())
    + " "
    + str(_get_contract_symbol(tbd_nordea))
)

In [ ]:
## get TBD DNB contract

tbd_dnb = _get_tbd_contract("TBD DNB")

# print TBD balance of Bob (retail investor at DNB Bank)
(
    str(tbd_dnb.functions.balanceOf(c.PK_BOB_TBD.address).call())
    + " "
    + str(_get_contract_symbol(tbd_dnb))
)

In [ ]:
tbd_nordea.functions.allowlistQuery(c.PK_ALICE_TBD.address).call(
    {"from": c.PK_NORDEA.address}
)

In [ ]:
tbd_dnb.functions.allowlistQuery(c.PK_BOB_TBD.address).call({"from": c.PK_DNB.address})

In [ ]:
## do a cct transfer
_do_contract_tx(
    tbd_nordea,
    "cctFrom",
    (c.PK_ALICE_TBD.address, c.PK_BOB_TBD.address, tbd_dnb.address, 10),
    c.PK_ALICE_TBD.key,
)

In [ ]:
(
    str(tbd_nordea.functions.balanceOf(c.PK_ALICE_TBD.address).call())
    + " "
    + str(_get_contract_symbol(tbd_nordea))
)

In [ ]:
(
    str(tbd_dnb.functions.balanceOf(c.PK_BOB_TBD.address).call())
    + " "
    + str(_get_contract_symbol(tbd_dnb))
)

### 2.2 DvP interactions

In [ ]:
## helper functions


def _get_dvp_contract():
    address = global_registry.functions.getContract("Delivery vs Payment").call()
    with open("contracts/DvP.sol/DvP.abi", "r") as f:
        abi = "".join(f.read().splitlines())
    return w3.eth.contract(address=address, abi=abi)


def _get_stocktokenfactory_contract(contractName: str):
    address = global_registry.functions.getContract(contractName).call()
    with open("contracts/StockTokenFactory.sol/StockTokenFactory.abi", "r") as f:
        abi = "".join(f.read().splitlines())
    return w3.eth.contract(address=address, abi=abi)


def _get_stocktoken_contract(contractName: str):
    stocktokenfactory = _get_stocktokenfactory_contract("StockToken Factory")

    found, address = stocktokenfactory.functions.getDeployedStockToken(
        "NO0001234567"
    ).call()
    assert found

    with open("contracts/StockToken.sol/StockToken.abi", "r") as f:
        abi = "".join(f.read().splitlines())
    return w3.eth.contract(address=address, abi=abi)


_settle_role = Web3.keccak(text="SETTLE_ROLE")

In [ ]:
## get DvP contract

dvp = _get_dvp_contract()

In [ ]:
## set this role for testing only, in a real setup, this user should not have that right
_do_contract_tx(
    dvp, "grantRole", (_settle_role, c.PK_NORGES_BANK.address), c.PK_NORGES_BANK.key
)

In [ ]:
## get a stock token contract

stocktoken = _get_stocktoken_contract("NO0001234567")

In [ ]:
init_stocktokens_alice = stocktoken.functions.balanceOf(c.PK_ALICE_SEC.address).call()
print(
    "Init stocktoken Alice: "
    + str(init_stocktokens_alice)
    + " "
    + str(_get_contract_symbol(stocktoken))
)
init_stocktokens_bob = stocktoken.functions.balanceOf(c.PK_BOB_SEC.address).call()
print(
    "Init stocktoken Bob: "
    + str(init_stocktokens_bob)
    + " "
    + str(_get_contract_symbol(stocktoken))
)
init_tbd_alice = tbd_nordea.functions.balanceOf(c.PK_ALICE_TBD.address).call()
print(
    "Init TBD Alice: "
    + str(init_tbd_alice)
    + " "
    + str(_get_contract_symbol(tbd_nordea))
)
init_tbd_bob = tbd_dnb.functions.balanceOf(c.PK_BOB_TBD.address).call()
print("Init TBD Bob: " + str(init_tbd_bob) + " " + str(_get_contract_symbol(tbd_dnb)))
init_wnok_dnb = wnok.functions.balanceOf(c.PK_DNB.address).call()
print("Init wNOK DNB: " + str(init_wnok_dnb) + " " + str(_get_contract_symbol(wnok)))
init_wnok_nordea = wnok.functions.balanceOf(c.PK_NORDEA.address).call()
print(
    "Init wNOK Nordea: " + str(init_wnok_nordea) + " " + str(_get_contract_symbol(wnok))
)

In [ ]:
## do a normal stocktoken transfer
_do_contract_tx(stocktoken, "transfer", (c.PK_BOB_SEC.address, 10), c.PK_ALICE_SEC.key)

In [ ]:
## do a settlement
_do_contract_tx(
    dvp,
    "settle",
    (
        stocktoken.address,
        c.PK_ALICE_SEC.address,
        c.PK_BOB_SEC.address,
        1,
        c.PK_ALICE_TBD.address,
        c.PK_BOB_TBD.address,
        42,
        tbd_nordea.address,
        tbd_dnb.address,
    ),
    c.PK_NORGES_BANK.key,
)

In [ ]:
stocktokens_alice = stocktoken.functions.balanceOf(c.PK_ALICE_SEC.address).call()
print(
    "Stocktoken Alice: "
    + str(stocktokens_alice)
    + " "
    + str(_get_contract_symbol(stocktoken))
)
stocktokens_bob = stocktoken.functions.balanceOf(c.PK_BOB_SEC.address).call()
print(
    "Stocktoken Bob: "
    + str(stocktokens_bob)
    + " "
    + str(_get_contract_symbol(stocktoken))
)
tbd_alice = tbd_nordea.functions.balanceOf(c.PK_ALICE_TBD.address).call()
print("TBD Alice: " + str(tbd_alice) + " " + str(_get_contract_symbol(tbd_nordea)))
tbd_bob = tbd_dnb.functions.balanceOf(c.PK_BOB_TBD.address).call()
print("TBD Bob: " + str(tbd_bob) + " " + str(_get_contract_symbol(tbd_dnb)))
wnok_dnb = wnok.functions.balanceOf(c.PK_DNB.address).call()
print("wNOK DNB: " + str(wnok_dnb) + " " + str(_get_contract_symbol(wnok)))
wnok_nordea = wnok.functions.balanceOf(c.PK_NORDEA.address).call()
print("wNOK Nordea: " + str(wnok_nordea) + " " + str(_get_contract_symbol(wnok)))

In [ ]:
## do a small check:
if init_stocktokens_alice - 11 != stocktokens_alice:
    print("Stocktoken balance Alice does not match")

if init_stocktokens_bob + 11 != stocktokens_bob:
    print("Stocktoken balance Bob does not match")

if init_tbd_alice + 42 != tbd_alice:
    print("TBD balance Alice does not match")

if init_tbd_bob - 42 != tbd_bob:
    print("TBD balance Bob does not match")

if init_wnok_nordea + 42 != wnok_nordea:
    print("wNOK balance Nordea does not match")

if init_wnok_dnb - 42 != wnok_dnb:
    print("wNOK balance DNB does not match")

## 2.3 Order Book Interactions

In [ ]:
## helper functions


def _get_orderbook_contract():
    address = global_registry.functions.getContract("Order Book").call()
    with open("contracts/OrderBook.sol/OrderBook.abi", "r") as f:
        abi = "".join(f.read().splitlines())
    return w3.eth.contract(address=address, abi=abi)

In [ ]:
## get Order Book contract

orderbook = _get_orderbook_contract()

In [ ]:
orderbook.functions.getSellOrders().call({"from": c.PK_BROKER1.address})

In [ ]:
orderbook.functions.getBuyOrders().call({"from": c.PK_BROKER2.address})

In [ ]:
_do_contract_tx(
    orderbook,
    "sell",
    (
        stocktoken.address,
        1,
        100,
        c.PK_ALICE_SEC.address,
        c.PK_ALICE_TBD.address,
        tbd_nordea.address,
    ),
    c.PK_BROKER1.key,
)

In [ ]:
orderbook.functions.getSellOrders().call({"from": c.PK_BROKER1.address})

In [ ]:
_do_contract_tx(
    orderbook,
    "buy",
    (
        stocktoken.address,
        1,
        100,
        c.PK_BOB_SEC.address,
        c.PK_BOB_TBD.address,
        tbd_dnb.address,
    ),
    c.PK_BROKER2.key,
)

In [ ]:
stocktokens_alice = stocktoken.functions.balanceOf(c.PK_ALICE_SEC.address).call()
print(
    "Stocktoken Alice: "
    + str(stocktokens_alice)
    + " "
    + str(_get_contract_symbol(stocktoken))
)
stocktokens_bob = stocktoken.functions.balanceOf(c.PK_BOB_SEC.address).call()
print(
    "Stocktoken Bob: "
    + str(stocktokens_bob)
    + " "
    + str(_get_contract_symbol(stocktoken))
)
tbd_alice = tbd_nordea.functions.balanceOf(c.PK_ALICE_TBD.address).call()
print("TBD Alice: " + str(tbd_alice) + " " + str(_get_contract_symbol(tbd_nordea)))
tbd_bob = tbd_dnb.functions.balanceOf(c.PK_BOB_TBD.address).call()
print("TBD Bob: " + str(tbd_bob) + " " + str(_get_contract_symbol(tbd_dnb)))
wnok_dnb = wnok.functions.balanceOf(c.PK_DNB.address).call()
print("wNOK DNB: " + str(wnok_dnb) + " " + str(_get_contract_symbol(wnok)))
wnok_nordea = wnok.functions.balanceOf(c.PK_NORDEA.address).call()
print("wNOK Nordea: " + str(wnok_nordea) + " " + str(_get_contract_symbol(wnok)))

In [ ]:
## do a small check:
if init_stocktokens_alice - 12 != stocktokens_alice:
    print("Stocktoken balance Alice does not match")

if init_stocktokens_bob + 12 != stocktokens_bob:
    print("Stocktoken balance Bob does not match")

if init_tbd_alice + 142 != tbd_alice:
    print("TBD balance Alice does not match")

if init_tbd_bob - 142 != tbd_bob:
    print("TBD balance Bob does not match")

if init_wnok_nordea + 142 != wnok_nordea:
    print("wNOK balance Nordea does not match")

if init_wnok_dnb - 142 != wnok_dnb:
    print("wNOK balance DNB does not match")

In [ ]:
## The following should create a sell order, invalidate it by removing the allowlist entry of the seller, add a corresponding buy order
## which should remove the sell order again, so it should end up with 1 buy order in the order book.
_do_contract_tx(
    orderbook,
    "sell",
    (
        stocktoken.address,
        1,
        99,
        c.PK_ALICE_SEC.address,
        c.PK_ALICE_TBD.address,
        tbd_nordea.address,
    ),
    c.PK_BROKER1.key,
)

In [ ]:
_do_contract_tx(tbd_nordea, "remove", [c.PK_ALICE_TBD.address], c.PK_NORDEA.key)

In [ ]:
_do_contract_tx(
    orderbook,
    "buy",
    (
        stocktoken.address,
        1,
        99,
        c.PK_BOB_SEC.address,
        c.PK_BOB_TBD.address,
        tbd_dnb.address,
    ),
    c.PK_BROKER2.key,
)

In [ ]:
if len(orderbook.functions.getSellOrders().call({"from": c.PK_BROKER1.address})) != 0:
    print("Sell orders should be empty")

In [ ]:
if len(orderbook.functions.getBuyOrders().call({"from": c.PK_BROKER2.address})) != 1:
    print("Buy orders should be length 1")

In [ ]:
_do_contract_tx(tbd_nordea, "add", [c.PK_ALICE_TBD.address], c.PK_NORDEA.key)